# Sub-session S020-S029: Model Evaluation & Export
Let's look inside the model's brain, find out where it's making mistakes, and export it for our web backend.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os

# 1. Re-train the model quickly so this notebook is self-contained
df = pd.read_csv('../data/raw/crop_recommendation.csv')
crops = df['label'].unique()
crop_map = {crop: i for i, crop in enumerate(crops)}
reverse_crop_map = {i: crop for crop, i in crop_map.items()}
df['label_encoded'] = df['label'].map(crop_map)

train_df = df.sample(frac=0.8, random_state=42)
test_df = df.drop(train_df.index)
features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']

X_train, y_train = train_df[features], train_df['label_encoded']
X_test, y_test = test_df[features], test_df['label_encoded']

train_mean = X_train.mean()
train_std = X_train.std()

X_train_scaled = (X_train - train_mean) / train_std
X_test_scaled = (X_test - train_mean) / train_std

model = xgb.XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='mlogloss')
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

print("Model successfully re-trained!")

### 1. Feature Importance
Which soil or weather metric did the AI rely on the most to make its decisions?

In [ ]:
plt.figure(figsize=(10, 6))
xgb.plot_importance(model, importance_type='weight', title='Feature Importance', show_values=False)
plt.tight_layout()
plt.show()

### 2. The Confusion Matrix
This matrix shows Actual Crops on the Left, and what the AI *Predicted* on the Bottom. 
A perfect model has numbers ONLY on the diagonal line.

In [ ]:
# Convert numerical predictions back to crop names for readability
y_test_names = pd.Series(y_test).map(reverse_crop_map).values
y_pred_names = pd.Series(y_pred).map(reverse_crop_map).values

cm = pd.crosstab(y_test_names, y_pred_names, rownames=['Actual'], colnames=['Predicted'])

plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, cmap='Blues', fmt='g')
plt.title("Confusion Matrix: Actual vs Predicted Crops")
plt.show()

### 3. Exporting (Serialization)
We save the XGBoost model, the Scaling values (mean and std), and the Crop dictionary so our backend API can load them instantly without retraining.

In [ ]:
os.makedirs('../models', exist_ok=True)

# Save the XGBoost model to a JSON file
model.save_model('../models/xgboost_crop_model.json')

# Save the Preprocessing math (mean/std) and the label dictionary to a Pickle (.pkl) file
preprocessing_data = {
    'mean': train_mean.to_dict(),
    'std': train_std.to_dict(),
    'crop_map': reverse_crop_map
}

with open('../models/preprocessing_data.pkl', 'wb') as f:
    pickle.dump(preprocessing_data, f)

print("✅ Brain successfully exported to 'ml_experiments/models/'!")